# 02 · DriverVideo → ET raw alignment

Annotate the shared speedometer-panel transitions and fit Driver exact-PTS time to ET raw. This step saves mapping/QC only; step 10 composes it with step 01 and writes compact DriverVideo files.

In [1]:
import importlib
import sys
from pathlib import Path

from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the BIGPROJECT/helpers folder')

from helpers.project_config import load_project_config
from helpers.sync_quality import driver_et_alignment as alignment
from helpers.sync_quality import panel_annotation as panel
importlib.reload(panel)
importlib.reload(alignment)

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
PROJECT_NAME = CONFIG.project
DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
DRIVER_VIDEO_SUBFOLDER = CONFIG.value('modalities', 'driver_video_source_subfolder')
ET_SUBFOLDER = CONFIG.value('modalities', 'eyeball_source_subfolder')
ANNOTATION_CSV = CONFIG.path('files', 'panel_annotations_csv')
ALIGNMENT_CSV = CONFIG.path('files', 'driver_et_alignment_csv')
ISSUES_CSV = CONFIG.path('files', 'driver_et_issues_csv')
OVERWRITE = True
PARTICIPANTS = None
VISITS = None
DRIFT_WARNING_PPM = 2000
DRIFT_REJECT_PPM = 10000
OFFSET_ONLY_DRIFT_BOUND_PPM = 5000
MAX_PAIR_UNCERTAINTY_S = 0.5

In [2]:
jobs = panel.discover_panel_video_pairs(
    DATA_ROOT, project=PROJECT_NAME, participants=PARTICIPANTS, visits=VISITS,
    driver_subfolder=DRIVER_VIDEO_SUBFOLDER, et_subfolder=ET_SUBFOLDER,
)
display(jobs[[
    'pid', 'visit', 'driver_path', 'et_scene_path', 'discovery_status',
]])

Discovered 43 visit(s): 42 ready for paired annotation.
Missing or ambiguous videos remain in the discovery table for review.


,pid,visit,driver_path,et_scene_path,discovery_status
0,A7G5SM,VX,E:\test\time_shifted\A7G5SM\Driver Video\A7G5S...,E:\test\time_shifted\A7G5SM\ET\A7G5SM_VX_et_ra...,ready
1,ABVLGQ,VX,E:\test\time_shifted\ABVLGQ\Driver Video\ABVLG...,E:\test\time_shifted\ABVLGQ\ET\ABVLGQ_VX_et_ra...,ready
2,B9NYQ8,VX,E:\test\time_shifted\B9NYQ8\Driver Video\B9NYQ...,E:\test\time_shifted\B9NYQ8\ET\B9NYQ8_VX_et_ra...,ready
3,CRVWF9,VX,E:\test\time_shifted\CRVWF9\Driver Video\CRVWF...,E:\test\time_shifted\CRVWF9\ET\CRVWF9_VX_et_ra...,ready
4,CRYSEP,VX,E:\test\time_shifted\CRYSEP\Driver Video\CRYSE...,E:\test\time_shifted\CRYSEP\ET\CRYSEP_VX_et_ra...,ready
5,CU6DM4,VX,E:\test\time_shifted\CU6DM4\Driver Video\CU6DM...,E:\test\time_shifted\CU6DM4\ET\CU6DM4_VX_et_ra...,ready
6,DUJL2L,VX,E:\test\time_shifted\DUJL2L\Driver Video\DUJL2...,E:\test\time_shifted\DUJL2L\ET\DUJL2L_VX_et_ra...,ready
7,E4MKXT,VX,E:\test\time_shifted\E4MKXT\Driver Video\E4MKX...,E:\test\time_shifted\E4MKXT\ET\E4MKXT_VX_et_ra...,ready
8,EASFXM,VX,E:\test\time_shifted\EASFXM\Driver Video\EASFX...,E:\test\time_shifted\EASFXM\ET\EASFXM_VX_et_ra...,ready
9,FEDZAG,VX,E:\test\time_shifted\FEDZAG\Driver Video\FEDZA...,E:\test\time_shifted\FEDZAG\ET\FEDZAG_VX_et_ra...,ready


## Manual panel annotation

In [3]:
if 'tool' in globals():
    tool.close()
tool = panel.launch_panel_annotator(
    jobs, annotation_csv=ANNOTATION_CSV,
)

In [4]:
annotations = panel.load_panel_annotations(ANNOTATION_CSV)
display(panel.summarize_panel_annotations(
    jobs, ANNOTATION_CSV, annotations=annotations,
))

,project,pid,visit,segment_id,discovery_status,driver_panel_off,driver_panel_on,et_scene_panel_off,et_scene_panel_on,event_order_valid,anchors_complete
0,ImBrAid,A7G5SM,VX,main,ready,accepted,accepted,accepted,accepted,True,True
1,ImBrAid,ABVLGQ,VX,main,ready,accepted,accepted,accepted,accepted,True,True
2,ImBrAid,B9NYQ8,VX,main,ready,accepted,accepted,accepted,accepted,True,True
3,ImBrAid,CRVWF9,VX,main,ready,accepted,accepted,accepted,accepted,True,True
4,ImBrAid,CRYSEP,VX,main,ready,accepted,accepted,accepted,accepted,True,True
5,ImBrAid,CU6DM4,VX,main,ready,accepted,accepted,accepted,accepted,True,True
6,ImBrAid,DUJL2L,VX,main,ready,accepted,accepted,accepted,accepted,True,True
7,ImBrAid,E4MKXT,VX,main,ready,accepted,accepted,accepted,accepted,True,True
8,ImBrAid,EASFXM,VX,main,ready,accepted,accepted,accepted,accepted,True,True
9,ImBrAid,FEDZAG,VX,main,ready,accepted,not_visible,accepted,accepted,True,False


In [5]:
alignments, issues = alignment.build_driver_et_alignments(
    annotations, video_pairs=jobs,
    drift_warning_ppm=DRIFT_WARNING_PPM,
    drift_reject_ppm=DRIFT_REJECT_PPM,
    offset_only_drift_bound_ppm=OFFSET_ONLY_DRIFT_BOUND_PPM,
    max_pair_uncertainty_s=MAX_PAIR_UNCERTAINTY_S,
)
if OVERWRITE:
    alignment.save_driver_et_alignment_results(
        alignments, issues, alignment_csv=ALIGNMENT_CSV, issues_csv=ISSUES_CSV,
    )
display(alignments[[
    'project', 'pid', 'visit', 'mapping_status', 'anchor_count',
    'usable', 'quality_grade', 'review_required',
    'driver_to_et_raw_drift_ppm', 'mapping_reason', 'warnings',
]])
display(issues)

,project,pid,visit,mapping_status,anchor_count,usable,quality_grade,review_required,driver_to_et_raw_drift_ppm,mapping_reason,warnings
0,ImBrAid,A7G5SM,VX,affine,2,True,B,False,522.924392,two accepted event pairs passed mapping QC,
1,ImBrAid,ABVLGQ,VX,affine,2,True,B,False,507.60176,two accepted event pairs passed mapping QC,
2,ImBrAid,B9NYQ8,VX,affine,2,True,B,False,529.84245,two accepted event pairs passed mapping QC,
3,ImBrAid,CRVWF9,VX,affine,2,True,B,False,528.516717,two accepted event pairs passed mapping QC,
4,ImBrAid,CRYSEP,VX,affine,2,True,A,False,463.093432,two accepted event pairs passed mapping QC,
5,ImBrAid,CU6DM4,VX,affine,2,True,A,False,474.700879,two accepted event pairs passed mapping QC,
6,ImBrAid,DUJL2L,VX,affine,2,True,A,False,479.862923,two accepted event pairs passed mapping QC,
7,ImBrAid,E4MKXT,VX,affine,2,True,A,False,455.066149,two accepted event pairs passed mapping QC,
8,ImBrAid,EASFXM,VX,affine,2,True,A,False,460.496887,two accepted event pairs passed mapping QC,
9,ImBrAid,FEDZAG,VX,offset_only,1,True,C,True,NaN,one paired event; usable with scale fixed to 1...,single anchor cannot estimate clock drift; sca...


,project,pid,visit,segment_id,severity,code,source,event,message
0,ImBrAid,FEDZAG,VX,main,warning,single_anchor_offset_only,,panel_off,single anchor cannot estimate clock drift; sca...
1,ImBrAid,G93LYM,VX,main,warning,single_anchor_offset_only,,panel_on,single anchor cannot estimate clock drift; sca...
2,ImBrAid,GLRNU9,VX,main,warning,single_anchor_offset_only,,panel_off,single anchor cannot estimate clock drift; sca...
3,ImBrAid,HH5RBX,VX,main,warning,single_anchor_offset_only,,panel_on,single anchor cannot estimate clock drift; sca...
4,ImBrAid,HKJPM3,VX,main,warning,no_paired_anchor,,,no panel event has accepted annotations in bot...
5,ImBrAid,HNWSPB,VX,main,warning,low_confidence_anchor,,,at least one accepted anchor has low confidence
6,ImBrAid,JVC62S,VX,main,error,drift_exceeds_reject_threshold,,,absolute clock drift 824330.6 ppm exceeds reje...
7,ImBrAid,MUHKQF,VX,main,warning,single_anchor_offset_only,,panel_on,single anchor cannot estimate clock drift; sca...
8,ImBrAid,R8YB77,VX,main,error,drift_exceeds_reject_threshold,,,absolute clock drift 75583.9 ppm exceeds rejec...
9,ImBrAid,RZKLE8,VX,main,warning,single_anchor_offset_only,,panel_off,single anchor cannot estimate clock drift; sca...
